In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
import sys

#math and array operations
import numpy as np
import math
import pandas as pd

#data classes
import xarray as xr
import pickle

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#loading bar
from tqdm import tqdm

#datetime
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data")
dataType = "Radar_STNRatio_Constant"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Load Model Directory Class
Region = "TRACER"; Case = "WET"; spinup_hours = "0"
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_TRACER = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

Region = "Hawaii"; Case = "WET"; spinup_hours = "12"
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_Hawaii = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

Region = "PRECIP"; Case = "WET"; spinup_hours = "12"
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_PRECIP = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
##################
#FUNCTIONS

In [ ]:
## GetRadarMasks

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

def GetRadarMasks():
    RadarDataMask_PRECIP = RadarObservationMask_Class.LoadMaskData_MRMS(DirectoryManager, ModelData_PRECIP)
    RadarDataMask_TRACER = RadarObservationMask_Class.LoadMaskData_MRMS(DirectoryManager, ModelData_TRACER)
    RadarDataMask_Hawaii = RadarObservationMask_Class.LoadMaskData_MRMS(DirectoryManager, ModelData_Hawaii)
    return RadarDataMask_PRECIP, RadarDataMask_TRACER, RadarDataMask_Hawaii

In [ ]:
import geopandas as gpd
import zipfile
import tempfile
def ReadKMZ_ToGeoDataFrame(kmzFilePath):
    """
    Reads a KMZ file and returns a GeoDataFrame.

    "nexrad.kmz" file is located at "https://gis.ncdc.noaa.gov/kml/nexrad.kmz"
    (Shows up on google search: nexrad radar latitudes and longitudes")
    """

    with tempfile.TemporaryDirectory() as tmpDir:
        with zipfile.ZipFile(kmzFilePath, "r") as kmz:
            kmz.extractall(tmpDir)

        # find the KML file
        kmlFiles = [
            f for f in os.listdir(tmpDir)
            if f.lower().endswith(".kml")
        ]

        if len(kmlFiles) == 0:
            raise FileNotFoundError("No KML found inside KMZ")

        kmlPath = os.path.join(tmpDir, kmlFiles[0])

        gdf = gpd.read_file(kmlPath)

    gdf["lon"] = gdf.geometry.x
    gdf["lat"] = gdf.geometry.y
    return gdf

def FindSpecificRadar_ByName(NexRadLocations, nameString):
    rows = NexRadLocations[
        NexRadLocations["Name"].str.contains(
            nameString,
            case=False,
            na=False
        )
    ]
    return rows
    
def FindSpecificRadar_ByLatLonRange(NexRadLocations,
                                    latitudeArray,
                                    longitudeArray):
    """
    Find radars within the lat/lon extent of a grid.
    """

    latMin = float(latitudeArray.min()) 
    latMax = float(latitudeArray.max())
    lonMin = float(longitudeArray.min())
    lonMax = float(longitudeArray.max())

    rows = NexRadLocations[
        (NexRadLocations["lat"] >= latMin) &
        (NexRadLocations["lat"] <= latMax) &
        (NexRadLocations["lon"] >= lonMin) &
        (NexRadLocations["lon"] <= lonMax)
    ]

    return rows

def GetSelectedRadarLocations(rows):
    RadarLocations = list(
        zip(
            rows["lat"].values,
            rows["lon"].values
        )
    )
    return RadarLocations

def PlotRadarPoints(RadarDataMask,
               RadarLocations):
    
    plt.contourf(RadarDataMask.longitude,RadarDataMask.latitude,RadarDataMask)
    for RadarLocation in RadarLocations:
        plt.scatter(RadarLocation[1],RadarLocation[0],color='red')

In [ ]:
def MakeLatLonGrids(ModelData):
    
    longitudeGrid, latitudeGrid = np.meshgrid(
        ModelData.longitude,
        ModelData.latitude,
        indexing="ij"
    )
    
    longitudeGrid = xr.DataArray(
        longitudeGrid,
        dims=("longitude", "latitude"),
        coords={
            "longitude": ModelData.longitude,
            "latitude": ModelData.latitude,
        },
        name="longitude"
    )
    
    latitudeGrid = xr.DataArray(
        latitudeGrid,
        dims=("longitude", "latitude"),
        coords={
            "longitude": ModelData.longitude,
            "latitude": ModelData.latitude,
        },
        name="latitude"
    )
    
    return longitudeGrid,latitudeGrid

In [ ]:
def Calculate_XY_FromRadar(RadarLocation,
                          longitudeGrid, latitudeGrid):
    """
    Calculates local Cartesian coordinates (x, y) relative to a radar location
    using a tangent-plane approximation.

    x = R * cos(Lat0) * (Lon - Lon0)
    y = R * (Lat - Lat0)

    """

    Lat0, Lon0 = RadarLocation
    EarthRadius = 6371.0  # km

    # Convert to radians
    Lat0Rad = np.deg2rad(Lat0)

    dLon = np.deg2rad(longitudeGrid - Lon0)
    dLat = np.deg2rad(latitudeGrid  - Lat0)

    # Local Cartesian coordinates (km)
    xGrid = EarthRadius * np.cos(Lat0Rad) * dLon
    yGrid = EarthRadius * dLat

    xGrid.name = "x_from_radar"
    yGrid.name = "y_from_radar"

    xGrid.attrs["units"] = "km"
    yGrid.attrs["units"] = "km"

    return xGrid, yGrid


def CalculateRange(x,y):
    r = x**2 + y**2
    r = np.sqrt(r)
    return r

In [ ]:
def CalculateSNR_constant(rGrid, radarType):
    """
    SNR (Signal to noise ratio) = Z_hh - (Noise_1km + 20*log10(r)) > 0
    ==> Z_hh > (Noise_1km + 20*log10(r))
    """
    if radarType in ["NexRad","TRACER","Hawaii"]:
        Noise_1km = -42
    elif radarType in ["PRECIP_SPol","PRECIP"]:
        Noise_1km = -42.4
    SNR_constant = (Noise_1km+20*np.log10(rGrid))
    return SNR_constant

def Calculate_PRECIP(RadarLocation=(24.82, 120.91),
                     ModelData=ModelData_PRECIP,
                     RadarDataMask=RadarDataMask_PRECIP):
    [longitudeGrid,latitudeGrid] = MakeLatLonGrids(ModelData)
    [xGrid, yGrid] = Calculate_XY_FromRadar(RadarLocation,
                              longitudeGrid, latitudeGrid)
    rGrid = CalculateRange(xGrid,yGrid)
    rGrid = rGrid.where(RadarDataMask==True) #Apply RadarDataMask
    SNR_constant = CalculateSNR_constant(rGrid, radarType="PRECIP")
    return SNR_constant,rGrid

def Calculate_TRACERHawaii(RadarLocations,
                           ModelData=ModelData_TRACER,
                           RadarDataMask=RadarDataMask_TRACER):

    #Get Longitude Latitude Meshgrids
    [longitudeGrid,latitudeGrid] = MakeLatLonGrids(ModelData)

    #Calculate All Radius Grids
    rGridList = []
    for count, RadarLocation in enumerate(RadarLocations):

        [xGrid, yGrid] = Calculate_XY_FromRadar(RadarLocation,
                                                longitudeGrid,latitudeGrid)
        rGrid = CalculateRange(xGrid,yGrid)
        rGrid = rGrid.where(RadarDataMask==True) #Apply RadarDataMask
        rGridList.append(rGrid)

    #Combine All Radius Grids
    rGrid_combined = xr.concat(
        rGridList,
        dim="radar"
    ).min(
        dim="radar",
        skipna=True
    )

    SNR_constant = CalculateSNR_constant(rGrid_combined, radarType="NexRad")
    return SNR_constant,rGrid_combined

In [ ]:
def SaveSNR_constant(SNR_constant,regionName):
    
    fileName = f"SNR_constant_{regionName}.nc"
    filePath = os.path.join(outputDirectory, fileName)
    SNR_constant.to_netcdf(filePath, mode="w")
    print(f"Saved to: {filePath}")

def LoadSNR_constant(regionName):

    codeType = os.path.join("DataAnalysis", "Observation_Data")
    dataType = "Radar_STNRatio_Constant"
    outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
    
    fileName = f"SNR_constant_{regionName}.nc"
    filePath = os.path.join(outputDirectory, fileName)
    
    SNR_constant = xr.open_dataarray(filePath)
    print(f"Loaded From: {filePath}")
    return SNR_constant

In [ ]:
## PlotData

COASTLINE_FEATURE = cfeature.COASTLINE
BORDERS_FEATURE   = cfeature.BORDERS
STATES_FEATURE    = cfeature.STATES
# def PlotData(dataArray, levels=15, cmap="viridis"):

#     fig = plt.figure(figsize=(8, 6))
#     ax = plt.axes(projection=ccrs.PlateCarree())

#     cf = ax.contourf(
#         dataArray.longitude,
#         dataArray.latitude,
#         dataArray,#.transpose("latitude", "longitude"),
#         levels=levels,
#         cmap=cmap,
#         transform=ccrs.PlateCarree()
#     )

#     plt.colorbar(cf, ax=ax, label="Range (km)")
#     ax.coastlines()
#     ax.add_feature(cfeature.BORDERS, edgecolor="white",facecolor="none",)
#     ax.add_feature(cfeature.STATES, edgecolor="white",facecolor="none",)

#     plt.tight_layout()
#     return fig, ax

import cartopy.mpl.ticker as cticker

def PlotData(dataArray, levels=15, cmap="viridis"):

    fig = plt.figure(figsize=(8, 6))
    ax = plt.axes(projection=ccrs.PlateCarree())

    cf = ax.contourf(
        dataArray.longitude,
        dataArray.latitude,
        dataArray.T,  # assumes dims (latitude, longitude) OR broadcasting works
        levels=levels,
        cmap=cmap,
        transform=ccrs.PlateCarree()
    )

    plt.colorbar(cf, ax=ax)

    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, edgecolor="white", facecolor="none")
    ax.add_feature(cfeature.STATES,  edgecolor="white", facecolor="none")

    # --- Gridlines with labels ---
    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        linewidth=0.0,   # hide grid lines
        color="none",    # hide grid lines
        alpha=0.0
    )
    
    gl.top_labels = False
    gl.right_labels = False
    
    gl.xlabel_style = {"size": 10}
    gl.ylabel_style = {"size": 10}

    plt.tight_layout()
    return fig, ax



In [ ]:
##################
#CALCULATING

In [ ]:
[RadarDataMask_PRECIP, RadarDataMask_TRACER, RadarDataMask_Hawaii] = GetRadarMasks()

In [ ]:
#PRECIP
RadarLocation_PRECIP = (24.82, 120.91)
[SNR_constant_PRECIP,rGrid_PRECIP] = Calculate_PRECIP(RadarLocation=RadarLocation_PRECIP,
                                                      RadarDataMask=RadarDataMask_PRECIP)
SaveSNR_constant(SNR_constant_PRECIP,regionName="PRECIP")

In [ ]:
#TRACER
NexRadLocations = ReadKMZ_ToGeoDataFrame("nexrad.kmz")
rows = FindSpecificRadar_ByLatLonRange(NexRadLocations,
                                latitudeArray=ModelData_TRACER.latitude,
                                longitudeArray=ModelData_TRACER.longitude)
RadarLocations_TRACER = GetSelectedRadarLocations(rows)
# PlotPoints(RadarDataMask_TRACER,RadarLocations_TRACER)

[SNR_constant_TRACER,rGrid_TRACER] = Calculate_TRACERHawaii(RadarLocations=RadarLocations_TRACER,
                                                            ModelData=ModelData_TRACER,
                                                            RadarDataMask=RadarDataMask_TRACER)
SaveSNR_constant(SNR_constant_TRACER,regionName="TRACER")

In [ ]:
#Hawaii
NexRadLocations = ReadKMZ_ToGeoDataFrame("nexrad.kmz")
rows = FindSpecificRadar_ByLatLonRange(NexRadLocations,
                                latitudeArray=ModelData_Hawaii.latitude,
                                longitudeArray=ModelData_Hawaii.longitude)[1:]
RadarLocations_Hawaii = GetSelectedRadarLocations(rows)
# PlotPoints(RadarDataMask_Hawaii,RadarLocations_Hawaii)

[SNR_constant_Hawaii,rGrid_Hawaii] = Calculate_TRACERHawaii(RadarLocations=RadarLocations_Hawaii,
                                                            ModelData=ModelData_Hawaii,
                                                            RadarDataMask=RadarDataMask_Hawaii)
SaveSNR_constant(SNR_constant_Hawaii,regionName="Hawaii")

In [ ]:
###################################
#PLOTTING FUNCTIONS

In [ ]:
def PlotOutput(rGrid=rGrid_PRECIP,SNR_constant=SNR_constant_PRECIP):
    PlotData(rGrid)
    PlotData(SNR_constant,cmap='RdBu')

In [ ]:
def MakeTestPlots(ModelData=ModelData_PRECIP,
                  SNR_constant=SNR_constant_PRECIP,
                  RadarDataMask=RadarDataMask_PRECIP,
                  t=150,zlevel=15):

    refl1 = ModelData.GetDataTimestep_diag(t=t,varName="refl10cm",
                                             printout=False)
    refl1=refl1.where(RadarDataMask==True)
    refl2 = refl1.where(refl1 >= SNR_constant)
    refl2 = refl1.where(lambda x: x >= SNR_constant) #both work

    #HORIZONTAL
    PlotData(refl1.isel(nVertLevels=zlevel),cmap='turbo')
    PlotData(refl2.isel(nVertLevels=zlevel),cmap='turbo')

    #VERTICAL, LONGITUDE
    a = refl1.mean(dim="latitude")
    b = refl2.mean(dim="latitude")
    
    xr.concat(
        [a, b],
        dim=xr.DataArray(
            ["refl1", "refl2"],
            dims="panel",
            name="panel"
        )
    ).plot(
        col="panel",
        col_wrap=1,
        cmap="turbo",
        figsize=(6, 6)
    )

    #VERTICAL, LATITUDE
    a = refl1.mean(dim="longitude")
    b = refl2.mean(dim="longitude")
    
    xr.concat(
        [a, b],
        dim=xr.DataArray(
            ["refl1", "refl2"],
            dims="panel",
            name="panel"
        )
    ).plot(
        col="panel",
        col_wrap=1,
        cmap="turbo",
        figsize=(6, 6)
    )

In [ ]:
###################################
#PLOTTING

In [ ]:
PlotOutput(rGrid_PRECIP,SNR_constant_PRECIP)

In [ ]:
MakeTestPlots(ModelData_PRECIP,SNR_constant_PRECIP,
              RadarDataMask_PRECIP,
              t=150)

In [ ]:
PlotOutput(rGrid_TRACER,SNR_constant_TRACER)

In [ ]:
MakeTestPlots(ModelData_TRACER,SNR_constant_TRACER,
              RadarDataMask_TRACER,
              t=100)

In [ ]:
PlotOutput(rGrid_Hawaii,SNR_constant_Hawaii)

In [ ]:
MakeTestPlots(ModelData_Hawaii,SNR_constant_Hawaii,
              RadarDataMask_Hawaii,
              t=100)